# 抽取式问答模型（BERT + CMRC 2018）

## 项目概览
- **模型**：BERT-Base-Chinese (102M 参数)
- **任务**：抽取式问答 (Extractive QA / 阅读理解)
- **数据集**：CMRC 2018（中文机器阅读理解）
- **输入**：问题 + 文章
- **输出**：从文章中抽取答案的起始和结束位置
- **评估指标**：F1（Token 级匹配）+ EM（精确匹配）

## 与生成式问答的对比
| 对比项 | 抽取式 QA（本项目） | 生成式 QA（DuReaderQG） |
|--------|---------------------|-------------------------|
| 模型 | BERT（Encoder Only） | T5（Encoder-Decoder） |
| 任务 | Token Classification（预测 start/end 位置） | Seq2Seq（生成文本） |
| 答案来源 | 必须出现在原文中 | 可自由生成 |
| 评估 | F1 + EM | BLEU |

## 核心技术
- **Stride 分割**：文章超长时，用滑动窗口分块，相邻块有重叠
- **Offset Mapping**：记录 token 位置 → 原文字符位置的映射
- **N-best 搜索**：枚举前 N 个最优 start/end 组合，选置信度最高的

## 运行说明
- 点击 Cell 左侧的 ▶ 按钮执行每个 Cell
- 或使用快捷键 `Shift + Enter` 逐个运行
- 或点击菜单 `Cell → Run All` 一次性运行全部
- 整个流程耗时约 1-2 小时（GPU）

## 第一步：环境检查和库导入

In [ ]:
import os
import json
import random
import collections
import re
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoConfig,
    AutoTokenizer,
    BertPreTrainedModel,
    BertModel,
    AdamW,
    get_scheduler
)

# 任务	用哪个 AutoModel
# 文本分类	AutoModelForSequenceClassification
# 命名实体识别	AutoModelForTokenClassification
# 问答（抽取式）	AutoModelForQuestionAnswering  ← 本项目
# 翻译 / 摘要 / T5	AutoModelForConditionalGeneration
# GPT / LLaMA 生成	AutoModelForCausalLM

from tqdm.auto import tqdm

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 库导入成功")
print(f"✓ PyTorch 版本: {torch.__version__}")
print(f"✓ GPU 可用: {torch.cuda.is_available()}")
print(f"✓ MPS 可用: {torch.backends.mps.is_available()}")

## 第二步：数据加载

In [ ]:
# 设置路径
DATA_DIR = '../../data/cmrc2018/'
OUTPUT_DIR = './qa_model_output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, 'cmrc2018_train.json')
DEV_FILE   = os.path.join(DATA_DIR, 'cmrc2018_dev.json')
TEST_FILE  = os.path.join(DATA_DIR, 'cmrc2018_trial.json')

# 检查数据文件
for name, path in [('训练数据', TRAIN_FILE), ('验证数据', DEV_FILE), ('测试数据', TEST_FILE)]:
    print(f"{name}: {path}")
    print(f"  存在: {os.path.exists(path)}")
    if os.path.exists(path):
        print(f"  大小: {os.path.getsize(path) / 1024 / 1024:.1f} MB")

print(f"\n输出目录: {OUTPUT_DIR}")

## 第三步：定义数据集类

CMRC 2018 数据格式（SQuAD 兼容）：
```json
{
  "data": [
    {
      "title": "文章标题",
      "paragraphs": [
        {
          "context": "文章内容...",
          "qas": [
            {
              "id": "DEV_0_QUERY_0",
              "question": "问题？",
              "answers": [{"text": "答案", "answer_start": 5}]
            }
          ]
        }
      ]
    }
  ]
}
```
注意：`answer_start` 是**字符位置**，不是 token 位置。

In [ ]:
class CMRC2018(Dataset):
    """CMRC 2018 阅读理解数据集（SQuAD 兼容格式）"""

    def __init__(self, data_file):
        self.data = self.load_data(data_file)

    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
            idx = 0
            for article in json_data['data']:
                title   = article['title']
                context = article['paragraphs'][0]['context']
                for question in article['paragraphs'][0]['qas']:
                    q_id   = question['id']
                    ques   = question['question']
                    text   = [ans['text']         for ans in question['answers']]
                    starts = [ans['answer_start'] for ans in question['answers']]
                    Data[idx] = {
                        'id':       q_id,
                        'title':    title,
                        'context':  context,
                        'question': ques,
                        'answers':  {'text': text, 'answer_start': starts}
                    }
                    idx += 1
        return Data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

print("✓ CMRC2018 数据集类定义成功")

## 第四步：加载分词器和数据

In [ ]:
# ============ 模型选择配置 ============
# 选项说明：
# 1. 'bert-base-chinese'       - Google BERT（中文，推荐）⭐⭐⭐ (默认)
# 2. 'hfl/chinese-roberta-wwm-ext'  - 哈工大 RoBERTa（中文，效果更好）⭐⭐⭐
# 3. 'hfl/chinese-macbert-base'     - MacBERT（中文，推荐）⭐⭐⭐
# 4. '/path/to/local/model'    - 本地模型路径
#
# 修改下面这行来切换模型：
MODEL_CHOICE = 'bert-base-chinese'  # ⬅️ 修改这里选择模型

MODEL_CONFIG = {
    'bert-base-chinese': {
        'model_name': 'bert-base-chinese',
        'description': 'Google BERT-Base-Chinese (102M 参数)'
    },
    'hfl/chinese-roberta-wwm-ext': {
        'model_name': 'hfl/chinese-roberta-wwm-ext',
        'description': '哈工大 RoBERTa-wwm-ext（中文，效果更好）'
    },
    'hfl/chinese-macbert-base': {
        'model_name': 'hfl/chinese-macbert-base',
        'description': 'MacBERT-Base（中文，推荐）'
    }
}

if MODEL_CHOICE.startswith('/') or MODEL_CHOICE.startswith('./'):
    model_name = MODEL_CHOICE
    print(f"✓ 模型选择: {MODEL_CHOICE} (本地路径)")
elif MODEL_CHOICE in MODEL_CONFIG:
    model_name = MODEL_CONFIG[MODEL_CHOICE]['model_name']
    print(f"✓ 模型选择: {MODEL_CONFIG[MODEL_CHOICE]['description']}")
else:
    model_name = MODEL_CHOICE
    print(f"✓ 模型选择: {MODEL_CHOICE} (HuggingFace ID)")

print(f"\n加载分词器: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("✓ 分词器加载成功")

print("\n加载训练集...")
train_dataset = CMRC2018(TRAIN_FILE)
print(f"✓ 加载 {len(train_dataset)} 条训练数据")

print("\n加载验证集...")
dev_dataset = CMRC2018(DEV_FILE)
print(f"✓ 加载 {len(dev_dataset)} 条验证数据")

print("\n加载测试集...")
test_dataset = CMRC2018(TEST_FILE)
print(f"✓ 加载 {len(test_dataset)} 条测试数据")

## 第五步：查看数据样本

In [ ]:
# 查看前 3 条样本
for i in range(3):
    sample = train_dataset[i]
    print(f"\n=== 样本 {i+1} ===")
    print(f"ID:    {sample['id']}")
    print(f"标题:  {sample['title']}")
    print(f"问题:  {sample['question']}")
    print(f"文章:  {sample['context'][:120]}...")
    print(f"答案:  {sample['answers']['text']}")
    print(f"起始位置: {sample['answers']['answer_start']}")
    # 验证 answer_start 是字符位置
    ans_text = sample['answers']['text'][0]
    ans_start = sample['answers']['answer_start'][0]
    actual = sample['context'][ans_start: ans_start + len(ans_text)]
    print(f"验证（从原文切割）: '{actual}' ✓" if actual == ans_text else f"验证失败: '{actual}' ≠ '{ans_text}'")

## 第六步：超参数配置

In [ ]:
# 超参数配置
class Args:
    # 模型
    model_type       = 'bert'
    num_labels       = 2         # start / end 各一个 logit
    # 数据处理
    max_length       = 512       # question + context 的最大 token 长度
    stride           = 128       # 超长文本分块时相邻块的重叠 token 数
    max_answer_length= 50        # 预测答案的最大 token 数（过滤太长的候选）
    n_best           = 20        # N-best 搜索：保留前 n_best 个候选 start/end
    # 训练
    batch_size       = 4
    num_train_epochs = 3
    learning_rate    = 1e-5
    warmup_proportion= 0.1
    weight_decay     = 0.01
    adam_beta1       = 0.9
    adam_beta2       = 0.98
    adam_epsilon     = 1e-8
    seed             = 42
    output_dir       = OUTPUT_DIR

args = Args()

# 设置随机种子
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

# 设置设备
if torch.backends.mps.is_available():
    args.device = torch.device('mps')
elif torch.cuda.is_available():
    args.device = torch.device('cuda')
else:
    args.device = torch.device('cpu')

print("超参数配置:")
for k, v in vars(args).items():
    print(f"  {k}: {v}")

## 第七步：定义模型

**抽取式 QA 模型结构**：
```
输入: [CLS] question [SEP] context [SEP]
  ↓
BERT Encoder → [batch, seq_len, hidden_size]
  ↓
Linear(hidden → 2) → [batch, seq_len, 2]
  ↓
split → start_logits [batch, seq_len]
      → end_logits   [batch, seq_len]
  ↓
推理：N-best 搜索，选置信度最高的 (start, end) 组合
```

**损失函数**：两个独立的 CrossEntropyLoss 取平均：
> L = (L_start + L_end) / 2

In [ ]:
class BertForExtractiveQA(BertPreTrainedModel):
    """基于 BERT 的抽取式阅读理解模型"""

    def __init__(self, config, args):
        super().__init__(config)
        self.num_labels = args.num_labels
        # add_pooling_layer=False：不需要 [CLS] 向量，需要完整序列输出
        self.bert       = BertModel(config, add_pooling_layer=False)
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        # 每个 token 输出 2 维：[start_logit, end_logit]
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.post_init()

    def forward(self, batch_inputs, start_positions=None, end_positions=None):
        bert_output     = self.bert(**batch_inputs)
        sequence_output = self.dropout(bert_output.last_hidden_state)
        logits          = self.classifier(sequence_output)  # [B, seq_len, 2]

        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1).contiguous()  # [B, seq_len]
        end_logits   = end_logits.squeeze(-1).contiguous()    # [B, seq_len]

        loss = None
        if start_positions is not None and end_positions is not None:
            loss_fct = CrossEntropyLoss()
            start_loss = loss_fct(start_logits, start_positions)
            end_loss   = loss_fct(end_logits,   end_positions)
            loss = (start_loss + end_loss) / 2

        return loss, start_logits, end_logits

print("✓ BertForExtractiveQA 类定义成功")

## 第八步：创建 DataLoader

**关键：Stride 分割 + Offset Mapping**

当文章很长时，不能直接截断（会丢失答案）。解决方案：
- 用滑动窗口把文章切成多个 chunk，相邻 chunk 重叠 `stride` 个 token
- 训练时：把答案的**字符位置** → 转换为 **token 位置**（作为标签）
- 推理时：用 offset_mapping 把 **token 位置** → 还原为**字符位置**（切割原文得到答案）

In [ ]:
def to_device(args, batch_data):
    """将 batch 数据移到指定设备"""
    new_batch_data = {}
    for k, v in batch_data.items():
        if k == 'batch_inputs':
            new_batch_data[k] = {k_: v_.to(args.device) for k_, v_ in v.items()}
        else:
            new_batch_data[k] = torch.tensor(v).to(args.device)
    return new_batch_data


def get_dataLoader(args, dataset, tokenizer, mode='train', shuffle=False):
    """创建 DataLoader，包含 stride 分割和 offset mapping"""
    assert mode in ['train', 'valid', 'test']

    def train_collote_fn(batch_samples):
        """训练数据预处理：生成答案的 start/end token 位置标签"""
        batch_question, batch_context, batch_answers = [], [], []
        for sample in batch_samples:
            batch_question.append(sample['question'])
            batch_context.append(sample['context'])
            batch_answers.append(sample['answers'])

        # 分词：启用 stride 处理超长文本
        batch_inputs = tokenizer(
            batch_question,
            batch_context,
            max_length=args.max_length,
            truncation='only_second',         # 只截断 context，保留完整 question
            stride=args.stride,               # 相邻 chunk 的重叠 token 数
            return_overflowing_tokens=True,   # 超长时分成多个 chunk
            return_offsets_mapping=True,      # 返回 token → 字符 的映射
            padding='max_length',
            return_tensors='pt'
        )

        offset_mapping = batch_inputs.pop('offset_mapping')
        sample_mapping = batch_inputs.pop('overflow_to_sample_mapping')

        start_positions, end_positions = [], []

        for i, offset in enumerate(offset_mapping):
            sample_idx = sample_mapping[i]
            answer     = batch_answers[sample_idx]

            # 答案的字符范围
            start_char = answer['answer_start'][0]
            end_char   = start_char + len(answer['text'][0])

            # 找到 context 在 token 序列中的位置（sequence_id == 1）
            sequence_ids  = batch_inputs.sequence_ids(i)
            idx = 0
            while idx < len(sequence_ids) and sequence_ids[idx] != 1:
                idx += 1
            context_start = idx
            while idx < len(sequence_ids) and sequence_ids[idx] == 1:
                idx += 1
            context_end = idx - 1

            # 检查答案是否在当前 chunk 内（可能被分割到别的 chunk）
            if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
                start_positions.append(0)
                end_positions.append(0)
            else:
                # 字符位置 → token 位置
                idx = context_start
                while idx <= context_end and offset[idx][0] <= start_char:
                    idx += 1
                start_positions.append(idx - 1)

                idx = context_end
                while idx >= context_start and offset[idx][1] >= end_char:
                    idx -= 1
                end_positions.append(idx + 1)

        return {
            'batch_inputs':    batch_inputs,
            'start_positions': start_positions,
            'end_positions':   end_positions
        }

    def test_collote_fn(batch_samples):
        """验证/测试数据预处理：保留 offset_mapping 用于答案还原"""
        batch_id, batch_question, batch_context = [], [], []
        for sample in batch_samples:
            batch_id.append(sample['id'])
            batch_question.append(sample['question'])
            batch_context.append(sample['context'])

        batch_inputs = tokenizer(
            batch_question,
            batch_context,
            max_length=args.max_length,
            truncation='only_second',
            stride=args.stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding='max_length',
            return_tensors='pt'
        )

        offset_mapping = batch_inputs.pop('offset_mapping').numpy().tolist()
        sample_mapping = batch_inputs.pop('overflow_to_sample_mapping')

        example_ids = []
        for i in range(len(batch_inputs['input_ids'])):
            sample_idx = sample_mapping[i]
            example_ids.append(batch_id[sample_idx])

            # 只保留 context 部分的 offset，question/padding 部分设为 None
            sequence_ids = batch_inputs.sequence_ids(i)
            offset_mapping[i] = [
                o if sequence_ids[k] == 1 else None
                for k, o in enumerate(offset_mapping[i])
            ]

        return {
            'batch_inputs':  batch_inputs,
            'offset_mapping': offset_mapping,
            'example_ids':   example_ids
        }

    collote_fn = train_collote_fn if mode == 'train' else test_collote_fn
    return DataLoader(dataset, batch_size=args.batch_size, shuffle=shuffle, collate_fn=collote_fn)


print("创建 DataLoader...")
train_dataloader = get_dataLoader(args, train_dataset, tokenizer, mode='train', shuffle=True)
dev_dataloader   = get_dataLoader(args, dev_dataset,   tokenizer, mode='valid', shuffle=False)
test_dataloader  = get_dataLoader(args, test_dataset,  tokenizer, mode='test',  shuffle=False)

print(f"✓ 训练集批次数: {len(train_dataloader)}")
print(f"✓ 验证集批次数: {len(dev_dataloader)}")
print(f"✓ 测试集批次数: {len(test_dataloader)}")

## 第九步：加载模型

In [ ]:
print(f"加载 BERT 模型: {model_name}...")
config = AutoConfig.from_pretrained(model_name)
model  = BertForExtractiveQA.from_pretrained(model_name, config=config, args=args)
model  = model.to(args.device)

print(f"✓ 模型加载成功")
print(f"✓ 模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"✓ 模型所在设备: {next(model.parameters()).device}")

## 第十步：设置优化器和学习率调度器

In [ ]:
# bias 和 LayerNorm.weight 不应用权重衰减
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {
        'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        'weight_decay': args.weight_decay
    },
    {
        'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0
    }
]

t_total = len(train_dataloader) * args.num_train_epochs
args.warmup_steps = int(t_total * args.warmup_proportion)

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=args.learning_rate,
    betas=(args.adam_beta1, args.adam_beta2),
    eps=args.adam_epsilon
)

lr_scheduler = get_scheduler(
    'linear',
    optimizer,
    num_warmup_steps=args.warmup_steps,
    num_training_steps=t_total
)

print(f"优化器: AdamW")
print(f"  学习率: {args.learning_rate}")
print(f"  β1/β2: {args.adam_beta1}/{args.adam_beta2}")
print(f"  权重衰减: {args.weight_decay}")
print(f"\n学习率调度器: Linear with Warmup")
print(f"  总步数: {t_total}")
print(f"  预热步数: {args.warmup_steps}")

## 第十一步：定义训练和评估函数

**评估函数说明**：
- **F1**：预测答案与参考答案的 Token 级最长公共子序列（LCS）比率
- **EM（Exact Match）**：预测答案与参考答案完全一致（去标点后）
- **AVG**：(F1 + EM) / 2，用于选择最佳模型

In [ ]:
# ============ F1 / EM 评估函数 ============

def mixed_segmentation(in_str, rm_punc=False):
    """中英文混合分词（中文按字符，英文按单词）"""
    in_str = str(in_str).lower().strip()
    sp_char = ['-',':','_','*','^','/','\\','~','`','+','=',
               '，','。','：','？','！','"','"','；',''','《','》','……','·','、',
               '「','」','（','）','－','～','『','』']
    segs_out, temp_str = [], ''
    for char in in_str:
        if rm_punc and char in sp_char:
            continue
        if re.search(r'[\u4e00-\u9fa5]', char) or char in sp_char:
            if temp_str:
                segs_out.extend(temp_str.split())
                temp_str = ''
            segs_out.append(char)
        else:
            temp_str += char
    if temp_str:
        segs_out.extend(temp_str.split())
    return segs_out

def remove_punctuation(in_str):
    sp_char = ['-',':','_','*','^','/','\\','~','`','+','=',
               '，','。','：','？','！','"','"','；',''','《','》','……','·','、',
               '「','」','（','）','－','～','『','』']
    return ''.join(c for c in str(in_str).lower().strip() if c not in sp_char)

def find_lcs(s1, s2):
    """求最长公共子序列长度"""
    m = [[0] * (len(s2)+1) for _ in range(len(s1)+1)]
    mmax, p = 0, 0
    for i in range(len(s1)):
        for j in range(len(s2)):
            if s1[i] == s2[j]:
                m[i+1][j+1] = m[i][j] + 1
                if m[i+1][j+1] > mmax:
                    mmax, p = m[i+1][j+1], i+1
    return s1[p-mmax:p], mmax

def calc_f1_score(answers, prediction):
    f1_scores = []
    for ans in answers:
        ans_segs  = mixed_segmentation(ans, rm_punc=True)
        pred_segs = mixed_segmentation(prediction, rm_punc=True)
        _, lcs_len = find_lcs(ans_segs, pred_segs)
        if lcs_len == 0:
            f1_scores.append(0)
            continue
        precision = lcs_len / len(pred_segs)
        recall    = lcs_len / len(ans_segs)
        f1_scores.append(2 * precision * recall / (precision + recall))
    return max(f1_scores)

def calc_em_score(answers, prediction):
    return int(any(remove_punctuation(ans) == remove_punctuation(prediction) for ans in answers))

def evaluate_metrics(predictions, references):
    """计算 F1 和 EM"""
    f1 = em = total = skip = 0
    pred_dict = {d['id']: d['prediction_text'] for d in predictions}
    ref_dict  = {d['id']: d['answers']['text'] for d in references}
    for qid, answers in ref_dict.items():
        total += 1
        if qid not in pred_dict:
            skip += 1
            continue
        f1 += calc_f1_score(answers, pred_dict[qid])
        em += calc_em_score(answers, pred_dict[qid])
    f1_score = 100.0 * f1 / total
    em_score = 100.0 * em / total
    return {'f1': f1_score, 'em': em_score, 'avg': (f1_score + em_score) * 0.5,
            'total': total, 'skip': skip}

print("✓ F1/EM 评估函数定义成功")

In [ ]:
def train_loop(args, dataloader, model, optimizer, lr_scheduler, epoch, total_loss):
    """训练一个 epoch"""
    model.train()
    finish_batch_num = epoch * len(dataloader)
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1} Training')

    for step, batch_data in enumerate(progress_bar, start=1):
        batch_data = to_device(args, batch_data)
        outputs    = model(**batch_data)
        loss       = outputs[0]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'avg_loss': f'{total_loss / (finish_batch_num + step):.4f}'})

    return total_loss

print("✓ train_loop 函数定义成功")

In [ ]:
def test_loop(args, dataloader, dataset, model):
    """
    验证/测试阶段：N-best 搜索 + offset mapping 还原答案

    步骤：
    1. 收集所有 feature 的 example_id 和 offset_mapping
    2. 建立 example → features 映射（一个 example 可能被分割成多个 chunk）
    3. 前向传播，收集所有 start/end logits
    4. N-best 搜索：枚举最优 (start, end) 组合
    5. 用 offset_mapping 还原字符位置，切割原文得到答案
    """
    # 收集所有 feature 的 ID 和 offset_mapping
    all_example_ids   = []
    all_offset_mapping = []
    for batch_data in dataloader:
        all_example_ids    += batch_data['example_ids']
        all_offset_mapping += batch_data['offset_mapping']

    # 建立 example → features 映射
    example_to_features = collections.defaultdict(list)
    for idx, feature_id in enumerate(all_example_ids):
        example_to_features[feature_id].append(idx)

    # 前向传播，收集 logits
    start_logits_all, end_logits_all = [], []
    model.eval()
    with torch.no_grad():
        for batch_data in tqdm(dataloader, desc='Evaluating'):
            batch_data.pop('example_ids')
            batch_data.pop('offset_mapping')
            batch_data = to_device(args, batch_data)
            outputs    = model(**batch_data)
            start_logits_all.append(outputs[1].cpu().numpy())
            end_logits_all.append(outputs[2].cpu().numpy())

    start_logits_all = np.concatenate(start_logits_all)
    end_logits_all   = np.concatenate(end_logits_all)

    # 真实答案
    theoretical_answers = [
        {'id': dataset[i]['id'], 'answers': dataset[i]['answers']}
        for i in range(len(dataset))
    ]

    # 对每个 example 进行 N-best 搜索
    predicted_answers = []
    for s_idx in tqdm(range(len(dataset)), desc='Predicting'):
        example_id = dataset[s_idx]['id']
        context    = dataset[s_idx]['context']
        answers    = []

        for feature_index in example_to_features[example_id]:
            start_logit = start_logits_all[feature_index]
            end_logit   = end_logits_all[feature_index]
            offsets     = all_offset_mapping[feature_index]

            # 取前 n_best 个候选 start 和 end
            start_indexes = np.argsort(start_logit)[-1:-args.n_best-1:-1].tolist()
            end_indexes   = np.argsort(end_logit)[-1:-args.n_best-1:-1].tolist()

            for si in start_indexes:
                for ei in end_indexes:
                    if offsets[si] is None or offsets[ei] is None:
                        continue
                    if ei < si or ei - si + 1 > args.max_answer_length:
                        continue
                    answers.append({
                        'start': offsets[si][0],
                        'text':  context[offsets[si][0]: offsets[ei][1]],
                        'score': start_logit[si] + end_logit[ei]
                    })

        if answers:
            best = max(answers, key=lambda x: x['score'])
            predicted_answers.append({'id': example_id, 'prediction_text': best['text'],
                                      'answer_start': best['start']})
        else:
            predicted_answers.append({'id': example_id, 'prediction_text': '', 'answer_start': 0})

    return evaluate_metrics(predicted_answers, theoretical_answers)

print("✓ test_loop 函数定义成功")

## 第十二步：开始训练 ⭐⭐⭐

⚠️ **这一步需要较长时间**（约 1-2 小时，取决于 GPU）

每个 Epoch 结束后会在验证集上评估，并保存最优模型权重（按 AVG = (F1 + EM) / 2 选择）。

In [ ]:
# 记录训练历史
training_history = {
    'train_loss': [],
    'dev_f1':     [],
    'dev_em':     [],
    'dev_avg':    []
}

best_avg_score  = 0.0
best_model_path = os.path.join(OUTPUT_DIR, 'best_model.bin')
total_loss      = 0.0

print("开始训练...\n")
for epoch in range(args.num_train_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{args.num_train_epochs}")
    print(f"{'='*60}")

    # 训练
    total_loss = train_loop(args, train_dataloader, model, optimizer, lr_scheduler, epoch, total_loss)
    avg_loss   = total_loss / ((epoch + 1) * len(train_dataloader))
    print(f"\n平均训练损失: {avg_loss:.4f}")
    training_history['train_loss'].append(avg_loss)

    # 验证
    metrics = test_loop(args, dev_dataloader, dev_dataset, model)
    F1_score, EM_score, avg_score = metrics['f1'], metrics['em'], metrics['avg']
    print(f"\n验证集指标:")
    print(f"  F1:  {F1_score:.2f}")
    print(f"  EM:  {EM_score:.2f}")
    print(f"  AVG: {avg_score:.2f}")

    training_history['dev_f1'].append(F1_score)
    training_history['dev_em'].append(EM_score)
    training_history['dev_avg'].append(avg_score)

    # 保存最佳模型
    if avg_score > best_avg_score:
        best_avg_score = avg_score
        torch.save(model.state_dict(), best_model_path)
        print(f"\n✓ 保存最佳模型 (AVG: {best_avg_score:.2f})")

print("\n" + "="*60)
print("✅ 训练完成！")
print("="*60)

## 第十三步：绘制训练曲线

In [ ]:
epochs = range(1, args.num_train_epochs + 1)

fig, (ax_loss, ax_score) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('模型训练收敛曲线', fontsize=15, fontweight='bold')

# 左图：训练 Loss
ax_loss.plot(epochs, training_history['train_loss'], label='训练 Loss',
             marker='o', linewidth=2, markersize=8)
ax_loss.set_xlabel('Epoch', fontsize=12)
ax_loss.set_ylabel('Loss', fontsize=12)
ax_loss.set_title('训练 Loss 曲线', fontsize=13, fontweight='bold')
ax_loss.set_xticks(epochs)
ax_loss.legend(fontsize=11)
ax_loss.grid(True, alpha=0.3)

# 右图：F1 / EM / AVG
ax_score.plot(epochs, training_history['dev_f1'],  label='F1',  marker='o', linewidth=2, markersize=8)
ax_score.plot(epochs, training_history['dev_em'],  label='EM',  marker='s', linewidth=2, markersize=8)
ax_score.plot(epochs, training_history['dev_avg'], label='AVG', marker='^', linewidth=2, markersize=8)
ax_score.set_xlabel('Epoch', fontsize=12)
ax_score.set_ylabel('分数', fontsize=12)
ax_score.set_title('验证集 F1 / EM 曲线', fontsize=13, fontweight='bold')
ax_score.set_xticks(epochs)
ax_score.legend(fontsize=11)
ax_score.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ 曲线已保存到 {OUTPUT_DIR}training_curves.png")

## 第十四步：加载最佳模型并定义预测函数

In [ ]:
# 加载最佳模型
model.load_state_dict(torch.load(best_model_path, map_location=args.device))
model.eval()
print(f"✓ 加载最佳模型")
print(f"✓ 模型路径: {best_model_path}")

In [ ]:
def predict_answer(args, context, question, model, tokenizer):
    """
    对单个问题进行抽取式预测

    步骤：
    1. 分词（启用 stride 处理超长文本）
    2. 前向传播，得到每个 token 的 start/end logits
    3. N-best 搜索：枚举最优 (start, end) 组合
    4. 用 offset_mapping 还原到原文字符位置
    """
    inputs = tokenizer(
        question,
        context,
        max_length=args.max_length,
        truncation='only_second',
        stride=args.stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length',
        return_tensors='pt'
    )

    chunk_num      = inputs['input_ids'].shape[0]
    inputs.pop('overflow_to_sample_mapping')
    offset_mapping = inputs.pop('offset_mapping').numpy().tolist()

    # 只保留 context 部分的 offset
    for i in range(chunk_num):
        sequence_ids   = inputs.sequence_ids(i)
        offset_mapping[i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(offset_mapping[i])
        ]

    batch_inputs = {'batch_inputs': {k: v.to(args.device) for k, v in inputs.items()}}

    with torch.no_grad():
        _, pred_start_logits, pred_end_logits = model(**batch_inputs)
        start_logits = pred_start_logits.cpu().numpy()
        end_logits   = pred_end_logits.cpu().numpy()

    answers = []
    for fi in range(chunk_num):
        sl      = start_logits[fi]
        el      = end_logits[fi]
        offsets = offset_mapping[fi]

        start_indexes = np.argsort(sl)[-1:-args.n_best-1:-1].tolist()
        end_indexes   = np.argsort(el)[-1:-args.n_best-1:-1].tolist()

        for si in start_indexes:
            for ei in end_indexes:
                if offsets[si] is None or offsets[ei] is None:
                    continue
                if ei < si or ei - si + 1 > args.max_answer_length:
                    continue
                answers.append({
                    'start': offsets[si][0],
                    'text':  context[offsets[si][0]: offsets[ei][1]],
                    'score': sl[si] + el[ei]
                })

    if answers:
        best = max(answers, key=lambda x: x['score'])
        return {'prediction_text': best['text'], 'answer_start': best['start']}
    return {'prediction_text': '', 'answer_start': 0}

print("✓ predict_answer 函数定义成功")

## 第十五步：验证集上的预测示例

In [ ]:
print("\n" + "="*80)
print("验证集上的预测示例 (前 5 个样本)")
print("="*80)

for i in range(min(5, len(dev_dataset))):
    sample       = dev_dataset[i]
    question     = sample['question']
    context      = sample['context']
    ground_truth = sample['answers']['text']

    result = predict_answer(args, context, question, model, tokenizer)
    prediction = result['prediction_text']

    f1 = calc_f1_score(ground_truth, prediction)
    em = calc_em_score(ground_truth, prediction)

    print(f"\n示例 {i + 1}:")
    print(f"❓ 问题: {question}")
    print(f"📄 文章: {context[:100]}...")
    print(f"✓ 参考答案: {ground_truth}")
    print(f"🤖 预测答案: {prediction}")
    print(f"📊 F1: {f1:.4f} | EM: {em}")
    print("-" * 80)

## 第十六步：自定义输入的预测

In [ ]:
print("\n" + "="*80)
print("自定义输入预测示例")
print("="*80)

examples = [
    {
        'question': '乔丹打了多少个赛季？',
        'context':  '迈克尔·乔丹在NBA打了15个赛季。他在1984年进入NBA，期间曾两度退役，最终于2003年正式退役。'
    },
    {
        'question': '北京是哪个国家的首都？',
        'context':  '北京是中华人民共和国的首都，是全国的政治、文化和国际交往中心，也是历史文化名城。'
    },
    {
        'question': '西湖位于哪个城市？',
        'context':  '西湖，位于浙江省杭州市区西部，是中国大陆首批国家重点风景名胜区和中国十大风景名胜之一，也是首批全国文明风景旅游区。'
    }
]

for idx, ex in enumerate(examples):
    result = predict_answer(args, ex['context'], ex['question'], model, tokenizer)
    print(f"\n示例 {idx + 1}:")
    print(f"❓ 问题: {ex['question']}")
    print(f"📄 文章: {ex['context']}")
    print(f"🤖 答案: {result['prediction_text']}")
    print(f"   位置: 字符 {result['answer_start']}")

print("\n" + "="*80)

## 第十七步：在测试集上评估

In [ ]:
print("在测试集上评估...")
test_metrics = test_loop(args, test_dataloader, test_dataset, model)

print(f"\n测试集指标:")
print(f"  F1:    {test_metrics['f1']:.2f}")
print(f"  EM:    {test_metrics['em']:.2f}")
print(f"  AVG:   {test_metrics['avg']:.2f}")
print(f"  Total: {test_metrics['total']}")
print(f"  Skip:  {test_metrics['skip']}")

## 第十八步：模型保存

In [ ]:
# 保存完整模型和分词器
model_save_path = os.path.join(OUTPUT_DIR, 'final_model')
os.makedirs(model_save_path, exist_ok=True)

model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✓ 模型已保存到: {model_save_path}")

# 保存超参数
hyperparams_dict = {k: str(v) for k, v in vars(args).items()}
with open(os.path.join(OUTPUT_DIR, 'hyperparams.json'), 'w', encoding='utf-8') as f:
    json.dump(hyperparams_dict, f, indent=2, ensure_ascii=False)
print(f"✓ 超参数已保存到: {os.path.join(OUTPUT_DIR, 'hyperparams.json')}")

# 保存训练历史
with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w', encoding='utf-8') as f:
    json.dump(training_history, f, indent=2, ensure_ascii=False)
print(f"✓ 训练历史已保存到: {os.path.join(OUTPUT_DIR, 'training_history.json')}")

## 第十九步：最终总结

In [ ]:
print("\n" + "="*80)
print("✅ 项目总结")
print("="*80)
print(f"""
【模型信息】
  架构: BERT-Base-Chinese (102M 参数)
  任务: 抽取式问答 (Extractive QA)
  数据集: CMRC 2018 (中文机器阅读理解)
  设备: {args.device}

【关键技术】
  Stride 分割: max_length={args.max_length}, stride={args.stride}
  N-best 搜索: n_best={args.n_best}, max_answer_length={args.max_answer_length}
  评估指标: F1（Token 级）+ EM（精确匹配）

【训练配置】
  批大小: {args.batch_size}
  学习率: {args.learning_rate}
  总 Epoch: {args.num_train_epochs}
  优化器: AdamW (warmup {args.warmup_proportion*100:.0f}%)

【最终指标 (第 {args.num_train_epochs} Epoch)】
  验证集 F1:  {training_history['dev_f1'][-1]:.2f}
  验证集 EM:  {training_history['dev_em'][-1]:.2f}
  验证集 AVG: {training_history['dev_avg'][-1]:.2f}
  测试集 F1:  {test_metrics['f1']:.2f}
  测试集 EM:  {test_metrics['em']:.2f}

【输出文件】
  最佳模型权重: {best_model_path}
  完整模型目录: {model_save_path}
  训练曲线:    {os.path.join(OUTPUT_DIR, 'training_curves.png')}
  超参数:      {os.path.join(OUTPUT_DIR, 'hyperparams.json')}
  训练历史:    {os.path.join(OUTPUT_DIR, 'training_history.json')}

【下一步】
  1. 查看 training_curves.png 分析训练情况
  2. 用 predict_answer() 函数对自定义问题进行预测
  3. 加载保存的模型:
     from transformers import AutoTokenizer, AutoConfig
     tokenizer = AutoTokenizer.from_pretrained('{model_save_path}')
     config    = AutoConfig.from_pretrained('{model_save_path}')
     model     = BertForExtractiveQA.from_pretrained('{model_save_path}', config=config, args=args)
  4. 尝试更强的预训练模型:
     - hfl/chinese-roberta-wwm-ext
     - hfl/chinese-macbert-base
""")
print("="*80)
print("\n🎉 训练完成！祝使用愉快！")